In [ ]:
%pip install huggingface_hub gradio langchain-community duckduckgo-search==5.3.1b1 transformers torch==2.4.0 inflect edge-tts asyncio streaming-stt-nemo==0.2.0 -qU

In [ ]:
%%shell
curl -fsSL https://ollama.com/install.sh | sh

# Execute Ollama in the background in the Terminal

ollama serve &

In [ ]:
%%shell
ollama list

In [ ]:
%%shell
ollama pull mistral

In [ ]:
%%shell
apt-get install -y tesseract-ocr

In [ ]:
%pip install arxiv unstructured[pdf] unstructured poppler-utils tesseract qdrant-client webdataset -qU

In [ ]:
import os
import arxiv


In [ ]:
dirpath = "arxiv_papers"
if not os.path.exists(dirpath):
   os.makedirs(dirpath)

search = arxiv.Search(
  query = "LLM", # your query length is limited by ARXIV_MAX_QUERY_LENGTH which is 300 characters
  max_results = 10,
  sort_by = arxiv.SortCriterion.LastUpdatedDate, # you can also use SubmittedDate or Relevance
  sort_order = arxiv.SortOrder.Descending
)

In [ ]:
import time
from urllib.error import HTTPError

for result in search.results():
    while True:
        try:
            result.download_pdf(dirpath=dirpath)
            print(f"-> Paper id {result.get_short_id()} with title '{result.title}' is downloaded.")
            break
        except FileNotFoundError:
            print("File not found")
            break
        except HTTPError:
            print("Forbidden")
            break
        except ConnectionResetError as e:
            print("Connection reset by peer")
            time.sleep(5)

In [ ]:
from langchain.document_loaders import PyPDFLoader,DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [ ]:
papers = []
loader = DirectoryLoader(dirpath, glob="./*.pdf", loader_cls=PyPDFLoader)
papers = loader.load()
print("Total number of pages loaded:", len(papers)) # Total number of pages loaded: 410

# This merges all papes from all papers into single text block for chunking
full_text = ''
for paper in papers:
    full_text = full_text + paper.page_content

full_text = " ".join(l for l in full_text.splitlines() if l)
print(len(full_text))

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap  = 50
)

paper_chunks = text_splitter.create_documents([full_text])

In [ ]:
additional_documents = []
additional_docs_path = "additional_documents"
loader = DirectoryLoader(additional_docs_path, glob="./*.pdf", loader_cls=PyPDFLoader)
additional_documents = loader.load()
print("Total number of pages loaded:", len(additional_documents)) # Total number of pages loaded: 410

# This merges all papes from all papers into single text block for chunking
full_text = ''
for paper in additional_documents:
    full_text = full_text + paper.page_content

full_text = " ".join(l for l in full_text.splitlines() if l)
print(len(full_text))

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap  = 50
)

additional_documents_chunks = text_splitter.create_documents([full_text])

In [ ]:
total_documents = paper_chunks + additional_documents_chunks
print("Total number of documents:", len(total_documents))

In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings

In [ ]:
%pip install sentence-transformers -qU

In [ ]:
model_name = "sentence-transformers/all-mpnet-base-v2"
model_kwargs = {"device": "cuda"}

try:
    embeddings = HuggingFaceEmbeddings(model_name=model_name, model_kwargs=model_kwargs)
except Exception as ex:
    print("Exception: ", ex)

In [ ]:
from langchain.vectorstores import Qdrant

In [ ]:
vectordb = Qdrant.from_documents(
    total_documents,
    embeddings,
    path="Qdrant_Persistv10",
    collection_name="voice_assistant_documents",
)

In [ ]:
from langchain.chains import RetrievalQA
from langchain_community.llms import Ollama

retriever = vectordb.as_retriever()

llm = Ollama(model="mistral")

qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    verbose=True
)

In [ ]:
from huggingface_hub import InferenceClient
import gradio as gr
import random
import tempfile
import asyncio
from streaming_stt_nemo import Model
import edge_tts

In [ ]:
# Initialize ASR model
default_lang = "en"
engines = { default_lang: Model(default_lang) }

In [ ]:
def transcribe(audio):
    """Transcribes the audio file to text."""
    lang = "en"
    model = engines[lang]
    text = model.stt_file(audio)[0]
    return text


In [ ]:
def format_prompt(message, history):
    """Formats the prompt for the language model."""
    prompt = "<s>"
    for user_prompt, bot_response in history:
        prompt += f"[INST] {user_prompt} [/INST]"
        prompt += f" {bot_response}</s> "
    prompt += f"[INST] {message} [/INST]"
    return prompt

In [ ]:
def generate(prompt, history, temperature=0.9, max_new_tokens=512, top_p=0.95, repetition_penalty=1.0):
    """Generates a response from the language model."""
    temperature = float(temperature)
    if temperature < 1e-2:
        temperature = 1e-2
    top_p = float(top_p)

    generate_kwargs = dict(
        temperature=temperature,
        max_new_tokens=max_new_tokens,
        top_p=top_p,
        repetition_penalty=repetition_penalty,
        do_sample=True,
        seed=random.randint(0, 10**7),
    )

    formatted_prompt = format_prompt(prompt, history)

    output = llm.invoke(formatted_prompt, **generate_kwargs, stream=False, details=True, return_full_text=False)

    search_result = qa.run(prompt)

    # search_result = duckduckgo_search.run(prompt)

    if search_result:
        yield search_result
    else:
        yield "Sorry, I couldn't find any relevant information."

In [ ]:
async def respond(audio):
    """Handles the full pipeline: transcribe, generate response, and TTS."""
    try:
        # Transcribe audio to text
        user_text = transcribe(audio)

        # Generate response using the language model
        history = []
        response_generator = generate(user_text, history)
        response_text = ""
        for response in response_generator:
            response_text = response

        # Convert the text response to speech
        communicate = edge_tts.Communicate(response_text)
        with tempfile.NamedTemporaryFile(delete=False, suffix=".wav") as tmp_file:
            tmp_path = tmp_file.name
            await communicate.save(tmp_path)
        return response_text, tmp_path
    except Exception as e:
        return str(e), None

In [ ]:
additional_inputs = [
    gr.Slider(
        label="Temperature",
        value=0.9,
        minimum=0.0,
        maximum=1.0,
        step=0.05,
        interactive=True,
        info="Higher values produce more diverse outputs",
    ),
    gr.Slider(
        label="Max new tokens",
        value=512,
        minimum=64,
        maximum=1024,
        step=64,
        interactive=True,
        info="The maximum numbers of new tokens",
    ),
    gr.Slider(
        label="Top-p (nucleus sampling)",
        value=0.90,
        minimum=0.0,
        maximum=1,
        step=0.05,
        interactive=True,
        info="Higher values sample more low-probability tokens",
    ),
    gr.Slider(
        label="Repetition penalty",
        value=1.2,
        minimum=1.0,
        maximum=2.0,
        step=0.05,
        interactive=True,
        info="Penalize repeated tokens",
    )
]

customCSS = """
#component-7 { # this is the default element ID of the chat component
  height: 800px; # adjust the height as needed
  flex-grow: 1;
}
"""

In [ ]:
with gr.Blocks(css=customCSS) as demo:
    gr.Markdown("REDIVAC GPT - VOICE ASSISTANT")
    gr.Markdown("This app is built to showcase how LLMs work On-Premises and Locally Hosted Models with indexed PDF documents using RAG.")

    with gr.Row():
        input_audio = gr.Audio(label="Voice Chat", sources="microphone", type="filepath", waveform_options=False)
        output_text = gr.Textbox(label="Text Response")
        output_audio = gr.Audio(label="REDIVAC", type="filepath", interactive=False, autoplay=True, elem_classes="audio")
        gr.Interface(fn=respond, inputs=[input_audio], outputs=[output_text, output_audio], live=True)

    with gr.Accordion("Settings", open=False):
        gr.Markdown("## Additional Parameters")

        for slider in additional_inputs:
            slider.render()

In [ ]:
demo.queue().launch(debug=True)